# PyTorch Mastery Course (Core Modules + Practice)

This notebook contains a modular, practice-focused curriculum to master PyTorch fundamentals used by senior ML/AI engineers and researchers. Each module includes: concept, why it matters, key points, and practice exercises with starter code. Work module-by-module and run cells in order.

---
## Course Structure (10 Modules)
1. Tensor fundamentals
2. Autograd & backprop
3. `torch.nn` and common layers
4. Training loop, losses, optimizers
5. Data pipeline & augmentation
6. CNNs & vision primitives
7. Sequence models & attention
8. Regularization, normalization, initialization
9. Performance & scaling (mixed precision, profiling)
10. Debugging, reproducibility, experiment tracking, deployment
---

## Module 1 — Tensor Fundamentals
**Concept:** `torch.Tensor` is an n‑dimensional array. Key attributes: `shape`, `dtype`, `device`, `ndim`.
**Why it matters:** All data and model parameters are tensors. Correct shapes/dtypes/devices are essential to avoid runtime errors and to get reproducible, efficient code.

**Key points**:
- Shape and dims (e.g., `(C,H,W)` vs `(H,W,C)`)
- Dtypes: floats for gradients (`float32`), integers for labels/indices, `uint8` for raw images.
- Device: `cpu` vs `cuda`. Move tensors with `.to(device)`.
- Indexing and slicing: `a[0, :]`, `a[:, -1]`, negative indices.
- Reshape/unsqueeze/squeeze/permute for ordering dims.
- Broadcasting rules for element-wise ops.

**Practice (starter tasks):** run and modify the code cell below to experiment.

In [ ]:
# Module 1 starters
import torch
import matplotlib.pyplot as plt
# 1. shapes and broadcasting examples
a = torch.tensor([1,2,3])        # shape (3,)
b = torch.tensor([[10],[20],[30]]) # shape (3,1)
print('a', a.shape, 'b', b.shape)
print('a + b ->', (a + b).shape)   # broadcasts to (3,3)

# 2. image CHW -> HWC and display (tiny random image)
img_chw = torch.randint(0,256,(3,4,4), dtype=torch.uint8)
img_hwc = img_chw.permute(1,2,0)  # H,W,C
print('img_chw', img_chw.shape, '-> img_hwc', img_hwc.shape)
plt.figure(figsize=(3,3)); plt.imshow(img_hwc.numpy()); plt.axis('off')

# 3. convert uint8 image to float normalized, move to device and back
device = 'cuda' if torch.cuda.is_available() else 'cpu'
img_f = img_hwc.float() / 255.0
img_f = img_f.to(device)
print('dtype,device', img_f.dtype, img_f.device)
img_back = img_f.cpu().numpy()

# 4. indexing examples
a2 = torch.arange(12).reshape(3,4)
print('a2=
', a2)
print('last column:', a2[:, -1])
print('first two rows:', a2[0:2, :])
print('reshape (2,6):', a2.reshape(2,6))

# 5. fix shape error example (broadcasting)
x = torch.rand(3,1)
y = torch.rand(4,)
# to add, make y shape (1,4) then broadcast to (3,4)
y2 = y.unsqueeze(0)   # shape (1,4)
res = x + y2          # broadcasts -> (3,4)
print('fixed shapes:', x.shape, y2.shape, '->', res.shape)

---
## Module 2 — Autograd & Backprop
**Concept:** Automatic differentiation builds a dynamic graph and computes gradients with `.backward()`.
**Why it matters:** Training uses gradients to update parameters; understanding autograd prevents common mistakes (e.g., forgetting `zero_grad`, mixing `.detach()` incorrectly).

**Key points:** `requires_grad`, `.grad`, `.grad_fn`, `with torch.no_grad()` for inference, `optimizer.zero_grad()` -> `loss.backward()` -> `optimizer.step()`.

**Practice:** small experiments below.

In [ ]:
# Module 2 starters
import torch
# 1. basic scalar autograd
x = torch.tensor([2.0], requires_grad=True)
y = x**3 + 2*x
y.backward()
print('x.grad should be dy/dx = 3*x^2 + 2 at x=2 ->', x.grad)

# 2. manual linear regression step (no nn)
# data: y = 2*x + 1
X = torch.tensor([[1.0],[2.0],[3.0]])
Y = torch.tensor([[3.0],[5.0],[7.0]])
w = torch.tensor([[0.0]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)
lr = 0.1
# forward
pred = X @ w + b
loss = ((pred - Y)**2).mean()
loss.backward()
# gradient step (manual)
with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()
print('updated w,b', w, b)

# 3. effect of no_grad on tracking
p = torch.tensor([1.0], requires_grad=True)
with torch.no_grad():
    q = p * 2
print('q has grad_fn?', hasattr(q, 'grad_fn'))

---
## Module 3 — `torch.nn` and Layers
**Concept:** `nn.Module` bundles parameters and forward logic.
**Why it matters:** Most models are composed of reusable modules (layers). Using `nn` makes code cleaner and interoperable with utilities like `torch.optim`.

**Key points:** `nn.Linear`, `nn.Conv2d`, `nn.BatchNorm`, `nn.Dropout`, creating custom `nn.Module`, `state_dict`.

**Practice starters:** implement small modules below.

In [ ]:
# Module 3 starters
import torch
import torch.nn as nn

# 1. 2-layer MLP
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return self.net(x)

m = MLP(4, 16, 2)
print('params count', sum(p.numel() for p in m.parameters() if p.requires_grad))

# 2. custom scaled linear layer
class ScaledLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.lin = nn.Linear(in_f, out_f)
        self.scale = nn.Parameter(torch.tensor(1.0))
    def forward(self, x):
        return self.lin(x) * self.scale

sl = ScaledLinear(3,1)
print('scaled linear params', sum(p.numel() for p in sl.parameters()))

---
## Module 4 — Training Loop, Losses, Optimizers
**Concept:** Training loop: forward -> loss -> backward -> optimizer.step().
**Why it matters:** This is the core cycle to fit models to data.

**Key points:** common losses (`CrossEntropyLoss`, `MSELoss`), optimizers (`SGD`, `Adam`), LR schedulers, `model.train()` vs `model.eval()`.

**Practice starters:** small training loop template below.

In [ ]:
# Module 4 starter: tiny training loop (synthetic data)
import torch, torch.nn as nn, torch.optim as optim
# synthetic regression: y = 3*x + 2
X = torch.randn(100,1)
Y = 3*X + 2 + 0.1*torch.randn_like(X)
dataset = torch.utils.data.TensorDataset(X,Y)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)
model = nn.Linear(1,1)
opt = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for epoch in range(5):
    model.train()
    running = 0.0
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        running += loss.item() * xb.size(0)
    print(f'Epoch {epoch}, loss', running / len(dataset))

---
## Module 5 — Data Pipeline & Augmentation
**Concept:** Use `Dataset` + `DataLoader` to feed batched data efficiently.
**Why it matters:** Real training requires streaming, shuffling, augmentation, and efficient IO.

**Key points:** implement `__len__`, `__getitem__`, use `num_workers`, `pin_memory`, `collate_fn` for variable-length items. Use `torchvision.transforms` for vision.

**Practice starters:** see code cell.

In [ ]:
# Module 5 starters: custom Dataset + transformations
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random

class SimpleCSVLike(Dataset):
    def __init__(self, n=100):
        self.X = torch.randn(n, 3)
        self.y = (self.X.sum(dim=1) > 0).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ds = SimpleCSVLike(200)
loader = DataLoader(ds, batch_size=16, shuffle=True, num_workers=0)
for xb, yb in loader:
    print('batch', xb.shape, yb.shape)
    break

# torchvision transforms example (PIL based normally) omitted here for brevity

---
## Module 6 — CNNs & Vision Primitives
**Concept:** Convolutions detect local patterns; pooling reduces spatial resolution.
**Why it matters:** Most vision models rely on convolutions; understanding shapes and receptive fields is essential for model design.

**Practice starters:** code below computes conv output shapes and builds a small CNN.

In [ ]:
# Module 6 starters: conv shape and tiny CNN
import torch, torch.nn as nn
x = torch.randn(1, 3, 32, 32)  # batch, C, H, W
conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
y = conv(x)
print('conv out shape', y.shape)

# tiny CNN
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        self.head = nn.Linear(32, 10)
    def forward(self, x):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        return self.head(f)

m = TinyCNN()
print('tiny cnn params', sum(p.numel() for p in m.parameters()))

---
## Module 7 — Sequence Models & Attention
**Concept:** RNNs/LSTMs/GRUs and Transformers process sequences; attention computes weighted sums across positions.

**Practice starters:** embedding + LSTM forward; simple scaled dot-product attention shape check.

In [ ]:
# Module 7 starters
import torch, torch.nn as nn
# embedding + LSTM
vocab_size, embed_dim = 50, 16
emb = nn.Embedding(vocab_size, embed_dim)
lstm = nn.LSTM(embed_dim, 32, batch_first=True)
x = torch.randint(0, vocab_size, (4, 10))  # batch, seq_len
x_emb = emb(x)
out, (h,c) = lstm(x_emb)
print('lstm out', out.shape)

# scaled dot-product attention (shapes)
Q = torch.randn(2,5,8)
K = torch.randn(2,6,8)
V = torch.randn(2,6,12)
scores = torch.matmul(Q, K.transpose(-2,-1)) / (8**0.5)  # (2,5,6)
weights = torch.softmax(scores, dim=-1)
attn = torch.matmul(weights, V)  # (2,5,12)
print('attn shape', attn.shape)

---
## Module 8 — Regularization, Normalization, Initialization
**Concepts & practice:** dropout, weight decay, BatchNorm/LayerNorm, `torch.nn.init`.
Try swapping normalizations and reinitializing weights in small models to observe training stability.

In [ ]:
# Module 8 starters: init and dropout test
import torch, torch.nn as nn
m = nn.Sequential(nn.Linear(10,20), nn.ReLU(), nn.Dropout(0.5), nn.Linear(20,1))
# init
for name, p in m.named_parameters():
    if 'weight' in name:
        nn.init.kaiming_normal_(p)

# forward with dropout behavior in train vs eval
x = torch.randn(4,10)
m.train(); print('train output mean', m(x).mean().item())
m.eval(); print('eval output mean', m(x).mean().item())

---
## Module 9 — Performance & Scaling
**Concepts:** mixed precision (`torch.cuda.amp`), gradient accumulation, profiling, multi-GPU basics.
**Practice starters:** small amp example below (requires GPU).

In [ ]:
# Module 9 starter: mixed precision example (if GPU available)
import torch, torch.nn as nn, torch.optim as optim
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = nn.Sequential(nn.Linear(1024,1024), nn.ReLU(), nn.Linear(1024,10)).to(device)
opt = optim.SGD(model.parameters(), lr=0.01)
scaler = torch.cuda.amp.GradScaler() if device.startswith('cuda') else None
x = torch.randn(16,1024).to(device)
y = torch.randint(0,10,(16,)).to(device)
loss_fn = nn.CrossEntropyLoss()
if scaler is not None:
    with torch.cuda.amp.autocast():
        logits = model(x)
        loss = loss_fn(logits, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
else:
    logits = model(x); loss = loss_fn(logits, y); loss.backward(); opt.step()
print('done step')

---
## Module 10 — Debugging, Reproducibility, Experiments
**Concepts & practice:** seeds for reproducibility, deterministic flags, logging metrics, creating small failing tests and fixing them.

**Practice starter:** set seeds and show run determinism (best-effort, note GPU nondeterminism caveats).

In [ ]:
# Module 10 starter: reproducibility
import torch, random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# quick check: two random tensors should be identical if seed set
a = torch.rand(3,3)
torch.manual_seed(SEED)
b = torch.rand(3,3)
print('equal?', torch.allclose(a,b))

---
## Projects & Next Steps
- Mini project A: CIFAR‑10 classifier (Modules 3–6).
- Mini project B: Fine-tune ResNet on small custom dataset (Modules 5–6).
- Mini project C: Small Transformer on toy seq2seq (Modules 7,8).

**If you want I can:** expand any module into a lesson-by-lesson notebook with full runnable exercises and solutions. Tell me which module to expand first.